# 第 48 章 · DDPM：一步步加噪声，再学着一步步去噪

配套讲义：[在线版](https://wikieden.github.io/beyond-transformer/lessons/48-ddpm.html) · 仓库里 `beyond-transformer/notes/lessons/48-ddpm.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 前向两步 | 式 2 | `forward_step(x, beta, eps)` | 0.8 → 0.896 → 0.746 |
| 第二步 一步跳 | 式 4 | `schedule()`、`jump(x0, t, eps)` | $\bar\alpha_2$ = 0.9702；t = 500 时系数 0.280、0.960 |
| 第三步 训练 | 式 14 | `train_example(0.8, 500, -0.5, -0.4)` | $x_{500}$ = −0.256，loss = 0.01，$\hat x_0$ ≈ 0.458 |
| 第四步 生成一步 | Algorithm 2 | `reverse_step(xt, 500, eps_hat, z=0)` | −0.2528 |

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

**函数 `schedule`**：论文的设置：β 线性从 0.0001 涨到 0.02；α = 1 − β；ᾱ = α 连乘。

In [ ]:
def schedule(T=1000, beta_1=1e-4, beta_T=0.02):
    """论文的设置：β 线性从 0.0001 涨到 0.02；α = 1 − β；ᾱ = α 连乘。"""
    betas = np.linspace(beta_1, beta_T, T)
    alphas = 1 - betas
    return betas, alphas, np.cumprod(alphas)

**函数 `forward_step`**：式 2：先缩 √(1 − β)，再加 √β · ε。

In [ ]:
def forward_step(x, beta, eps):
    """式 2：先缩 √(1 − β)，再加 √β · ε。"""
    return np.sqrt(1 - beta) * x + np.sqrt(beta) * eps

**函数 `jump`**：式 4：x_t = √ᾱ_t · x0 + √(1 − ᾱ_t) · ε（t 从 1 数）。

In [ ]:
def jump(x0, t, eps, alpha_bar):
    """式 4：x_t = √ᾱ_t · x0 + √(1 − ᾱ_t) · ε（t 从 1 数）。"""
    ab = alpha_bar[t - 1]
    return np.sqrt(ab) * x0 + np.sqrt(1 - ab) * eps

**函数 `train_example`**：Algorithm 1 的一次：造 x_t，算猜噪声的平方误差，并用猜的噪声反推 x0。

In [ ]:
def train_example(x0, t, eps, eps_hat, alpha_bar):
    """Algorithm 1 的一次：造 x_t，算猜噪声的平方误差，并用猜的噪声反推 x0。"""
    ab = alpha_bar[t - 1]
    xt = jump(x0, t, eps, alpha_bar)
    x0_hat = (xt - np.sqrt(1 - ab) * eps_hat) / np.sqrt(ab)
    return xt, (eps - eps_hat) ** 2, x0_hat

**函数 `reverse_step`**：Algorithm 2 的一步：减一小部分猜出的噪声、放大回来、再加 σ_t·z（σ_t² = β_t）。

In [ ]:
def reverse_step(xt, t, eps_hat, z, betas, alphas, alpha_bar):
    """Algorithm 2 的一步：减一小部分猜出的噪声、放大回来、再加 σ_t·z（σ_t² = β_t）。"""
    i = t - 1
    mean = (xt - betas[i] / np.sqrt(1 - alpha_bar[i]) * eps_hat) / np.sqrt(alphas[i])
    return mean + np.sqrt(betas[i]) * z

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
betas, alphas, alpha_bar = schedule()
print("\n第一步 前向两步：x0 = 0.8，β = 0.01、0.02，ε = 1.0、−1.0")
x1 = forward_step(0.8, 0.01, 1.0)
x2 = forward_step(x1, 0.02, -1.0)
print(round(x1, 3), round(x2, 3))

In [ ]:
print("\n第二步 一步跳：ᾱ_2 = 0.99 × 0.98 =", round(0.99 * 0.98, 4))
for t in (1, 100, 250, 500, 750, 1000):
    ab = alpha_bar[t - 1]
    print(f"t = {t:4d}：ᾱ = {ab:.5f}，原图系数 {np.sqrt(ab):.3f}，噪声系数 {np.sqrt(1 - ab):.3f}")

In [ ]:
print("\n第三步 训练一次：x0 = 0.8，t = 500，ε = −0.5，网络猜 −0.4")
xt, loss, x0_hat = train_example(0.8, 500, -0.5, -0.4, alpha_bar)
print(f"x_500 = {xt:.3f}，loss = {loss:.3f}，反推 x0 ≈ {x0_hat:.3f}")

In [ ]:
print("\n第四步 生成一步：从 x_500 到 x_499，z 取 0")
print(round(reverse_step(xt, 500, -0.4, 0.0, betas, alphas, alpha_bar), 4))

## 3. 代码里才看得到的

- 论文的 t 从 1 数到 1000，数组下标从 0 数，所以第 t 步是 `betas[t - 1]`。
- `np.cumprod` 一次算出全部 1000 个 $\bar\alpha_t$，训练时直接查表，不用真的加 t 次噪声。

## 4. 自己改一改

- 把 β 的上限从 0.02 改成 0.01，看第 1000 步的原图系数是不是还接近 0。
- 在 `reverse_step` 里把 z 改成 1，看随机项让结果挪了多少。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 墨水入水散得快，倒着收回要靠学。
> 每步先缩再加噪，方差守住还是一。
> 阿尔法连乘一步跳，原图系数降、噪声升。
> 随机挑步撒噪声，网络专猜撒了啥。
> 生成从噪声倒着走，每步减一点、再撒一点，一千步才成图。